# EDA -- fertilizer_prediction clean.csv (R5)

Exploration only. Nothing under `src/` imports from this notebook -- conclusions are exported to `ml/data/EDA_FINDINGS.md` for anything Saloni or the pipeline actually needs. Run top to bottom on `ml/data/processed/clean.csv` (produced by `python -m src.data_pipeline.clean`).

In [ ]:
import pandas as pd

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 20)

df = pd.read_csv('../data/processed/clean.csv')
df.shape

## Dtypes, cardinality, missingness

In [ ]:
df.dtypes

In [ ]:
df.nunique()

In [ ]:
df.isna().sum()  # expect all zero -- clean.py's schema already enforces this

## Distributions per crop and per fertilizer

50 rows across 5 crops (chickpea has zero rows -- see the R4 note on the `Pulses` label gap) and 7 fertilizer products.

In [ ]:
df['crop_id'].value_counts()

In [ ]:
df['product_id'].value_counts()

In [ ]:
df.groupby('crop_id')[['temperature_c', 'humidity_pct', 'moisture_pct']]\
  .agg(['min', 'max', 'std']).round(1)

## Correlation and collinearity

Among the numeric features actually used by the classifier (temperature, humidity, moisture, and the unitless N/P/K columns).

In [ ]:
numeric_cols = ['temperature_c', 'humidity_pct', 'moisture_pct',
                'nitrogen_raw', 'potassium_raw', 'phosphorous_raw']
df[numeric_cols].corr().round(2)

**temperature_c and humidity_pct are r=0.98 -- near-total collinearity.** Carrying both as separate features is close to carrying one feature twice. nitrogen_raw is also meaningfully anti-correlated with phosphorous_raw (r=-0.71) and potassium_raw (r=-0.52), consistent with a fixed-total-nutrient-budget-style generation pattern rather than independently measured field samples.

## Target relationships

Is the fertilizer label predictable from crop alone, or from crop+soil_type alone? If so, N/P/K/climate features would be redundant and the classifier would just be learning a lookup table.

In [ ]:
pd.crosstab(df['crop_id'], df['product_id'])

In [ ]:
# distinct products seen per crop -- if this were 1, product would be fully
# determined by crop alone
df.groupby('crop_id')['product_id'].nunique()

In [ ]:
# distinct products seen per (crop, soil_type) -- if this were 1 everywhere,
# product would be fully determined by crop+soil_type, and N/P/K would carry
# no additional signal for the classifier
df.groupby(['crop_id', 'soil_type'])['product_id'].nunique()

**Good news: product is not determined by crop or by (crop, soil_type) alone.** Every (crop, soil_type) group sees 3+ distinct products, so N/P/K genuinely carries discriminating signal -- the classifier can't just memorize a crop-to-fertilizer lookup table and call it done.

## Group structure -- are rows independent?

Check whether soil_type is itself redundant with crop_id (a structural, non-independent relationship baked into how this dataset was built, not a modelling concern about the target but about feature redundancy).

In [ ]:
df.groupby('crop_id')['soil_type'].unique()

**wheat, maize and rice each appear with exactly one soil_type** (Loamy, Sandy, Clayey respectively) -- `soil_type` is 100% redundant with `crop_id` for 3 of 5 crops. Only cotton and sugarcane vary soil_type independently of crop. This isn't target leakage, but it means a feature-importance read on `soil_type` may really be echoing `crop_id` for most of the data, worth remembering when interpreting the trained model later.

## Duplicates that would straddle a future train/test split

Exact duplicates first, then near-duplicates (climate features within +-1, same crop+soil_type) -- the more realistic risk at this dataset size.

In [ ]:
df.duplicated().sum()  # exact full-row duplicates

In [ ]:
feat = ['temperature_c', 'humidity_pct', 'moisture_pct']
pairs = []
for (c, s), g in df.groupby(['crop_id', 'soil_type']):
    g = g.reset_index(drop=True)
    for i in range(len(g)):
        for j in range(i + 1, len(g)):
            if all(abs(g.loc[i, f] - g.loc[j, f]) <= 1 for f in feat):
                pairs.append((c, s, i, j, g.loc[i, 'product_id'], g.loc[j, 'product_id']))
pairs

**3 near-duplicate pairs found.** Two (both in `rice`/`Clayey`) have *different* product labels despite near-identical climate readings -- reassuring, it confirms climate alone doesn't leak the label, N/P/K is doing real work. But the third pair (`sugarcane`/`Loamy`, rows with nearly identical temperature/humidity/moisture **and the same label** `np_20_20_0`) is a genuine leakage risk: if a naive random split puts one of that pair in train and the other in test, the test score on that row is inflated by near-memorization, not generalization. With only 50 rows this is one pair out of a handful of test rows -- small in absolute terms, but large relative to how few test rows exist to measure the real accuracy against.

## Leakage red flags -- summary

- No exact duplicate rows (0/50).
- No feature or feature-pair fully determines the label (ruled out crop-alone and   crop+soil_type-alone as trivial predictors).
- `soil_type` is redundant with `crop_id` for 3/5 crops -- a collinearity note, not   target leakage.
- `temperature_c`/`humidity_pct` are r=0.98 -- treat as effectively one feature.
- One same-label near-duplicate pair (sugarcane/Loamy) is a real, if small,   train/test contamination risk at this dataset size.

## What this data can and cannot support

**Label balance:** 7 classes, 50 rows. Majority class (`urea`) is 11/50 = **22% majority-baseline accuracy**. The rarest class (`npk_10_26_26`) has only **2 total rows** -- it cannot appear in train, val *and* test simultaneously under any split, and can't be meaningfully learned at all with 1-2 examples.

**Is accuracy above the majority baseline plausible?** Likely yes in principle -- the target-relationship analysis above shows real, non-redundant signal in N/P/K. But *reporting* a precise accuracy number is not reliable at this size: a 70/15/15 split leaves roughly 7-8 test rows, and a proportion estimated from 7-8 samples has a binomial 95% CI of roughly +-35 percentage points. Any single-split accuracy figure should be treated as noise, not a real estimate, until bootstrap CIs (R8's planned `evaluate_classifier`) or cross-validation are applied.

**Recommendation for Saloni:**
1. **Model family:** favor low-variance, strongly regularized models over a    many-tree XGBoost, which will overfit badly on ~35 training rows. A regularized    multinomial logistic regression or a shallow, heavily-constrained tree/small    XGBoost (few estimators, max_depth 2-3, strong L2) are more appropriate starting    points. Always compare against the majority-class and stratified-random baselines    (`baseline_report` in the planned metrics module) -- with data this small, a    complex model beating a trivial baseline is not guaranteed.
2. **CV scheme:** don't trust a single frozen train/val/test split as the primary    evidence at n=50 -- use stratified k-fold cross-validation (k=5, or leave-one-out    given how small this is) and report the mean +- std across folds. The frozen test    split contract (R6's `test_ids.json`) can still exist for reproducibility, but its    single-split number should be presented as a rough check, not the headline metric.
3. **Merge or exclude rare classes:** `npk_10_26_26` (2 rows) should either be merged    into a compositionally similar NPK-complex group or excluded from the classifier    target entirely, letting the rule-based `reference_doses.csv`/`soil_adjustments.csv`    layer pick a nutrient-matched product instead of asking the classifier to learn a    near-unlearnable 2-example class. `npk_17_17_17` (5 rows) is thin too -- not    unusable, but worth watching in the per-class metrics.

Full writeup: `ml/data/EDA_FINDINGS.md`.